In [1]:
import pandas as pd
import numpy as np
import ast
import warnings
from pathlib import Path
from functools import reduce
import lightgbm as lgb
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

BASE       = Path("/mnt/c/etri-lifelog/data/raw/data")
SENSOR_DIR = BASE / "ch2025_data_items"
TARGETS    = ["Q1","Q2","Q3","S1","S2","S3","S4"]

train = pd.read_csv(BASE / "ch2026_metrics_train.csv")
sub   = pd.read_csv(BASE / "ch2026_submission_sample.csv")

train["lifelog_date"] = pd.to_datetime(train["lifelog_date"])
train["sleep_date"]   = pd.to_datetime(train["sleep_date"])

print(f"train: {train.shape}, test: {sub.shape}")
print(f"train 기간: {train.lifelog_date.min()} ~ {train.lifelog_date.max()}")

train: (450, 10), test: (250, 10)
train 기간: 2024-06-03 00:00:00 ~ 2024-11-14 00:00:00


In [2]:
def load_simple(fname, value_col):
    df = pd.read_parquet(SENSOR_DIR / fname)
    df["date"]  = df["timestamp"].dt.date.astype(str)
    df["hour"]  = df["timestamp"].dt.hour

    # 전체 집계
    g = df.groupby(["subject_id","date"])[value_col]
    base = g.agg(["mean","std","min","max","count"]).reset_index()
    base.columns = ["subject_id","lifelog_date"] + \
                   [f"{value_col}_{s}" for s in ["mean","std","min","max","count"]]

    # 밤 (22~06시)
    night = df[df["hour"].isin(list(range(22,24)) + list(range(0,7)))]
    ng = night.groupby(["subject_id","date"])[value_col].agg(
        **{f"{value_col}_night_mean":"mean", f"{value_col}_night_std":"std"}
    ).reset_index().rename(columns={"date":"lifelog_date"})
    base = base.merge(ng, on=["subject_id","lifelog_date"], how="left")

    # 낮 (07~21시)
    day = df[df["hour"].isin(range(7,22))]
    dg = day.groupby(["subject_id","date"])[value_col].agg(
        **{f"{value_col}_day_mean":"mean", f"{value_col}_day_std":"std"}
    ).reset_index().rename(columns={"date":"lifelog_date"})
    base = base.merge(dg, on=["subject_id","lifelog_date"], how="left")

    return base


def load_hr():
    df = pd.read_parquet(SENSOR_DIR / "ch2025_wHr.parquet")
    df["heart_rate"] = df["heart_rate"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df = df.explode("heart_rate")
    df["heart_rate"] = pd.to_numeric(df["heart_rate"], errors="coerce")
    df = df.dropna(subset=["heart_rate"])
    df["date"] = df["timestamp"].dt.date.astype(str)
    df["hour"] = df["timestamp"].dt.hour

    base = df.groupby(["subject_id","date"])["heart_rate"].agg(
        hr_mean="mean", hr_std="std", hr_min="min", hr_max="max",
        hr_median=lambda x: x.median(),
        hr_q75=lambda x: x.quantile(0.75),
    ).reset_index().rename(columns={"date":"lifelog_date"})

    sleep  = df[df["hour"].isin(list(range(23,24)) + list(range(0,7)))]
    active = df[df["hour"].isin(range(7,23))]

    sg = sleep.groupby(["subject_id","date"])["heart_rate"].agg(
        hr_sleep_mean="mean", hr_sleep_std="std"
    ).reset_index().rename(columns={"date":"lifelog_date"})

    ag = active.groupby(["subject_id","date"])["heart_rate"].agg(
        hr_active_mean="mean", hr_active_std="std"
    ).reset_index().rename(columns={"date":"lifelog_date"})

    base = base.merge(sg, on=["subject_id","lifelog_date"], how="left")
    base = base.merge(ag, on=["subject_id","lifelog_date"], how="left")
    base["hr_sleep_active_diff"] = base["hr_active_mean"] - base["hr_sleep_mean"]

    return base


def load_pedo():
    df = pd.read_parquet(SENSOR_DIR / "ch2025_wPedo.parquet")
    df["date"] = df["timestamp"].dt.date.astype(str)
    agg = df.groupby(["subject_id","date"]).agg(
        step_sum=("step","sum"),
        step_mean=("step","mean"),
        distance_sum=("distance","sum"),
        speed_mean=("speed","mean"),
        calories_sum=("burned_calories","sum"),
        running_sum=("running_step","sum"),
    ).reset_index().rename(columns={"date":"lifelog_date"})
    return agg


# 로딩
feat_ac     = load_simple("ch2025_mACStatus.parquet",     "m_charging")
feat_act    = load_simple("ch2025_mActivity.parquet",     "m_activity")
feat_light  = load_simple("ch2025_mLight.parquet",        "m_light")
feat_screen = load_simple("ch2025_mScreenStatus.parquet", "m_screen_use")
feat_wlight = load_simple("ch2025_wLight.parquet",        "w_light")
feat_hr     = load_hr()
feat_pedo   = load_pedo()

feats = [feat_ac, feat_act, feat_light, feat_screen, feat_wlight, feat_hr, feat_pedo]
feat_all = reduce(lambda a,b: pd.merge(a, b, on=["subject_id","lifelog_date"], how="left"), feats)

print(f"피처 테이블: {feat_all.shape}")

피처 테이블: (700, 64)


In [5]:
# train 조인
df = train.copy()
df["lifelog_date"] = df["lifelog_date"].astype(str)
feat_all["lifelog_date"] = feat_all["lifelog_date"].astype(str)
df = df.merge(feat_all, on=["subject_id","lifelog_date"], how="left")
df["lifelog_date"] = pd.to_datetime(df["lifelog_date"])
df = df.sort_values(["subject_id","lifelog_date"]).reset_index(drop=True)

# 누수 없는 개인화: row 이전 데이터만으로 expanding mean
SENSOR_COLS = [c for c in df.columns
               if c not in TARGETS + ["subject_id","sleep_date","lifelog_date"]]

personal_feats = []
for col in ["m_charging_mean","m_activity_mean","m_light_mean",
            "m_screen_use_mean","hr_mean","hr_sleep_mean","hr_active_mean",
            "step_sum","distance_sum"]:
    if col not in df.columns:
        continue
    # shift(1): 자기 자신 제외, 이전 데이터만
    prior_mean = df.groupby("subject_id")[col].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    df[f"{col}_prior_mean"] = prior_mean
    df[f"{col}_dev"] = df[col] - prior_mean
    personal_feats += [f"{col}_prior_mean", f"{col}_dev"]

df = df.fillna(0)

FEATURE_COLS = [c for c in df.columns
                if c not in TARGETS + ["subject_id","sleep_date","lifelog_date"]]

print(f"최종 df: {df.shape}, 피처 수: {len(FEATURE_COLS)}")

최종 df: (450, 90), 피처 수: 80


In [6]:
# 전체 날짜 기준 뒤 25% → val
all_dates = sorted(df["lifelog_date"].unique())
n_val     = int(len(all_dates) * 0.25)
val_dates = set(all_dates[-n_val:])

train_df = df[~df["lifelog_date"].isin(val_dates)]
val_df   = df[ df["lifelog_date"].isin(val_dates)]

print(f"train: {train_df.shape}, val: {val_df.shape}")
print(f"val 기간: {min(val_dates).date()} ~ {max(val_dates).date()}")
print(f"train subjects: {train_df.subject_id.nunique()}, val subjects: {val_df.subject_id.nunique()}")

train: (406, 90), val: (44, 90)
val 기간: 2024-09-25 ~ 2024-11-14
train subjects: 10, val subjects: 3


In [7]:
params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "n_estimators": 500,
    "early_stopping_rounds": 40,
    "verbose": -1,
    "random_state": 42,
}

models, val_losses = {}, {}

for target in TARGETS:
    model = lgb.LGBMClassifier(**params)
    model.fit(
        train_df[FEATURE_COLS], train_df[target],
        eval_set=[(val_df[FEATURE_COLS], val_df[target])],
        callbacks=[lgb.early_stopping(40, verbose=False),
                   lgb.log_evaluation(False)]
    )
    pred = model.predict_proba(val_df[FEATURE_COLS])[:, 1]
    loss = log_loss(val_df[target], pred)
    models[target]    = model
    val_losses[target] = loss
    print(f"{target}: {loss:.4f}")

avg = np.mean(list(val_losses.values()))
print(f"\n✅ Average Log-Loss (val): {avg:.4f}")

Q1: 0.7050
Q2: 0.6683
Q3: 0.6132
S1: 0.5902
S2: 0.6666
S3: 0.7376
S4: 0.7020

✅ Average Log-Loss (val): 0.6690


In [8]:
# test 피처 조인
sub_df = sub.copy()
sub_df["lifelog_date"] = pd.to_datetime(sub_df["lifelog_date"]).astype(str)
feat_all["lifelog_date"] = feat_all["lifelog_date"].astype(str)

sub_df = sub_df.merge(feat_all, on=["subject_id","lifelog_date"], how="left")
sub_df["lifelog_date"] = pd.to_datetime(sub_df["lifelog_date"])
sub_df = sub_df.sort_values(["subject_id","lifelog_date"]).reset_index(drop=True)

# test에도 누수 없는 개인화 피처 적용
# train의 마지막 prior_mean을 test에 이어붙이기
for col in ["m_charging_mean","m_activity_mean","m_light_mean",
            "m_screen_use_mean","hr_mean","hr_sleep_mean","hr_active_mean",
            "step_sum","distance_sum"]:
    if col not in sub_df.columns:
        continue
    # train에서 각 subject의 마지막 prior_mean을 가져와서 test 초기값으로 사용
    last_prior = df.groupby("subject_id")[f"{col}_prior_mean"].last()
    sub_df[f"{col}_prior_mean"] = sub_df["subject_id"].map(last_prior)
    sub_df[f"{col}_dev"] = sub_df[col] - sub_df[f"{col}_prior_mean"]

sub_df = sub_df.fillna(0)
X_test = sub_df[FEATURE_COLS]

print(f"test shape: {X_test.shape}, 결측: {X_test.isnull().sum().sum()}")

# 전체 train으로 재학습
final_models = {}
for target in TARGETS:
    model = lgb.LGBMClassifier(**{k:v for k,v in params.items()
                                   if k != "early_stopping_rounds"})
    model.fit(df[FEATURE_COLS], df[target])
    final_models[target] = model

# 제출 파일
submission = sub[["subject_id","sleep_date","lifelog_date"]].copy()
for target in TARGETS:
    submission[target] = final_models[target].predict_proba(X_test)[:, 1]

submission.to_csv("/mnt/c/etri-lifelog/submissions/lgbm_v4_global_split.csv", index=False)
print("✅ 저장 완료")
print(submission[TARGETS].describe().round(3))

test shape: (250, 80), 결측: 0
✅ 저장 완료
            Q1       Q2       Q3       S1       S2       S3       S4
count  250.000  250.000  250.000  250.000  250.000  250.000  250.000
mean     0.587    0.640    0.716    0.806    0.687    0.721    0.708
std      0.418    0.392    0.382    0.322    0.402    0.393    0.369
min      0.000    0.000    0.000    0.000    0.000    0.000    0.000
25%      0.072    0.196    0.446    0.767    0.291    0.373    0.440
50%      0.770    0.881    0.967    0.996    0.963    0.985    0.920
75%      0.992    0.991    0.998    1.000    0.999    0.999    0.997
max      1.000    1.000    1.000    1.000    1.000    1.000    1.000
